# FDCPA Rule Classifier — QLoRA Training

**Run this on Kaggle with GPU (T4 x1 or T4 x2) enabled. Internet must be ON.**

- Base model: Qwen/Qwen2.5-3B-Instruct
- Quantization: NF4 (BitsAndBytes)
- LoRA: rank=16, alpha=32
- Expected runtime: ~60–90 min on T4

**Kaggle setup:**
1. Upload `data/train.jsonl` and `data/val.jsonl` as a Kaggle dataset named `fdcpa-data`
2. Add Kaggle Secrets: `HF_TOKEN`, `WANDB_API_KEY`

In [ ]:
# Step 1: Clone repo and install dependencies
!git clone https://github.com/ree2raz/fdcpa-rule-classifier.git /kaggle/working/fdcpa-rule-classifier
!pip install -q transformers>=4.45.0 peft>=0.13.0 trl>=0.11.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets>=3.0.0 wandb>=0.18.0 python-dotenv tqdm scikit-learn matplotlib seaborn openai

import sys
sys.path.insert(0, '/kaggle/working/fdcpa-rule-classifier/src')
import os
os.chdir('/kaggle/working')

In [ ]:
# Step 2: Copy data files from Kaggle dataset into repo
# Assumes you uploaded data/ as a Kaggle dataset named 'fdcpa-data'
import shutil
from pathlib import Path

repo_data = Path('/kaggle/working/fdcpa-rule-classifier/data')
kaggle_input = Path('/kaggle/input/fdcpa-data')

# If data exists as Kaggle dataset, copy it into the repo
if kaggle_input.exists():
    for f in kaggle_input.glob('*.jsonl'):
        shutil.copy2(f, repo_data / f.name)
        print(f'Copied {f.name}')
else:
    print('No Kaggle dataset found — data must already be in the repo or generated first.')

# Verify data exists
for split in ['train.jsonl', 'val.jsonl']:
    p = repo_data / split
    if p.exists():
        with open(p) as f:
            count = sum(1 for line in f if line.strip())
        print(f'{split}: {count} examples')
    else:
        print(f'WARNING: {split} not found at {p}')

## 1. Set Environment Variables

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')

## 2. Initialize W&B

In [ ]:
import wandb
from datetime import datetime

run_name = f"qlora-r16-a32-{datetime.now().strftime('%Y%m%d-%H%M')}"
wandb.init(
    project='fdcpa-rule-classifier',
    name=run_name,
    config={
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'lora_rank': 16,
        'lora_alpha': 32,
        'epochs': 3,
        'batch_size': 4,
        'gradient_accumulation': 4,
        'learning_rate': 2e-4,
    },
)
print(f"W&B run: {run_name}")

## 3. Load Model and Tokenizer

In [ ]:
import torch
from fdcpa_classifier.train import load_model_and_tokenizer, get_lora_config

model, tokenizer = load_model_and_tokenizer()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"VRAM reserved:  {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## 4. Load and Tokenize Datasets

In [ ]:
from fdcpa_classifier.train import build_dataset

train_ds = build_dataset('train', tokenizer)
val_ds = build_dataset('val', tokenizer)

print(f"Train: {len(train_ds)} examples")
print(f"Val:   {len(val_ds)} examples")

sample = train_ds[0]['text']
print(f"\nSample training example (first 500 chars):")
print(sample[:500] + '...')

## 5. Configure LoRA

In [ ]:
from peft import get_peft_model

lora_config = get_lora_config()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
from fdcpa_classifier.train import get_training_args, MAX_SEQ_LENGTH
from trl import SFTTrainer

training_args = get_training_args()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
)

import time
start_time = time.time()
trainer.train()
training_time = time.time() - start_time

print(f"\nTraining completed in {training_time / 60:.1f} minutes")

## 7. Quality Checks

In [ ]:
eval_results = trainer.evaluate()
print(f"Final eval loss: {eval_results.get('eval_loss', 'N/A')}")

train_loss = trainer.state.log_history
final_train_loss = [x['loss'] for x in train_loss if 'loss' in x]
if final_train_loss:
    print(f"Final train loss: {final_train_loss[-1]:.4f}")

if torch.cuda.is_available():
    peak_vram = torch.cuda.max_memory_allocated(0) / 1e9
    print(f"Peak VRAM: {peak_vram:.2f} GB")
    assert peak_vram < 14, f"VRAM exceeded 14GB: {peak_vram:.2f}GB"

print("\nAll quality checks passed.")

## 8. Save Adapter

In [ ]:
from pathlib import Path

final_dir = Path('/kaggle/working/checkpoints/final')
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Adapter saved to {final_dir}")

## 9. Push to HuggingFace Hub

In [ ]:
hf_repo = 'ree2raz/fdcpa-rule-classifier-qlora'
model.push_to_hub(hf_repo, token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub(hf_repo, token=os.environ['HF_TOKEN'])
print(f"Pushed to https://huggingface.co/{hf_repo}")

In [ ]:
wandb.finish()
print(f"\nTotal training time: {training_time / 60:.1f} minutes")
print(f"W&B run name: {run_name}")
print("Done. Proceed to 03_eval_comparison.ipynb")